In [3]:
import pandas as pd
import numpy as np
import faiss

from pathlib import Path
from sentence_transformers import SentenceTransformer

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
PROJECT_PATH = Path(r"C:\Projects\Enterprise AI Analytics Platform for Customer Complaints")

In [9]:
index = faiss.read_index(
    str(PROJECT_PATH / "vector_db" / "complaints_faiss.index")
)

print(index.ntotal)

100000


In [11]:
metadata = pd.read_csv(
    PROJECT_PATH / "vector_db" / "complaints_metadata.csv"
)

metadata.head()

,complaint_id,final_text_for_embeddings
0,10866944,purchased vehicle company called car credit we...
1,12856369,recently logged mohela account find extra accr...
2,13160260,public service loan forgiveness program submit...
3,12979646,provided
4,12972549,xx xx scrub credit scored dropped point middle...


In [13]:
def retrieve_similar_complaints(query, top_k=5):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        results.append({
            "rank": rank,
            "complaint_id": metadata.iloc[idx]["complaint_id"],
            "distance": distances[0][rank - 1],
            "text": metadata.iloc[idx]["final_text_for_embeddings"]
        })

    return pd.DataFrame(results)

In [15]:
retrieve_similar_complaints(
    "my credit report has incorrect information",
    top_k=5
)

,rank,complaint_id,distance,text
0,1,20009658,0.538444,personal information credit report outdated
1,2,20372151,0.592833,credit report inaccurate inaccuracy causing cr...
2,3,21935653,0.592833,credit report inaccurate inaccuracy causing cr...
3,4,20362362,0.594021,recently reviewed consumer credit report disco...
4,5,21193233,0.625674,disputing inaccurate unverifiable information ...


In [17]:
def create_rag_context(query, top_k=5):
    retrieved_df = retrieve_similar_complaints(query, top_k)

    context = ""

    for _, row in retrieved_df.iterrows():
        context += f"""
Complaint ID: {row['complaint_id']}
Complaint Text: {row['text']}
---
"""

    return context

In [19]:
query = "my credit report has incorrect information"

context = create_rag_context(query, top_k=5)

print(context)


Complaint ID: 20009658
Complaint Text: personal information credit report outdated
---

Complaint ID: 20372151
Complaint Text: credit report inaccurate inaccuracy causing creditor deny credit duty report accurate information consumer please investigate account inquires update account accordingly avoid future litigation
---

Complaint ID: 21935653
Complaint Text: credit report inaccurate inaccuracy causing creditor deny credit duty report accurate information consumer please investigate account inquires update account accordingly avoid future litigation
---

Complaint ID: 20362362
Complaint Text: recently reviewed consumer credit report discovered inaccurate personal information belong information appears incorrect may reported error one company identifying issue took action reviewing credit file preparing formal request investigation correction inaccurate information may affecting accuracy credit report requesting reporting company investigate source information requesting company con

In [27]:
!pip install requests

In [29]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": "Explain RAG in one sentence.",
        "stream": False
    }
)

print(response.json()["response"])

RAG (Reactive Attachment Disorder) is a psychological disorder that affects children, characterized by an abnormal attachment style to caregivers due to early trauma or neglect, often resulting in difficulties with emotional regulation and relationships.


In [31]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": "Explain Retrieval Augmented Generation (RAG) in one sentence.",
        "stream": False
    }
)

print(response.json()["response"])

Retrieval Augmented Generation (RAG) is a machine learning approach that combines retrieval-based information search with generation models, such as language generators or text-to-text models, to produce high-quality text outputs by leveraging pre-trained retriever models and fine-tuning them on a specific task.


In [35]:
import requests

def ask_llm(context, question):
    prompt = f"""
You are a financial complaint assistant.

Context:
{context}

Question:
{question}

Answer only using the context above.
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json()["response"]

In [37]:
question = "What is the main problem in these credit report complaints?"

context = create_rag_context(question, top_k=5)

answer = ask_llm(context, question)

print(answer)

The main problem in these credit report complaints is inaccurate information reported on their credit files, which they believe resulted from a data breach and lack of proper notification. They are disputing various errors such as incorrect account balances, addresses, tradelines, and identity theft, and requesting immediate correction, removal, or deletion of the inaccuracies from their credit reports.


In [39]:
def ask_llm_with_prompt(context, question):
    prompt = f"""
You are an AI assistant for a financial services complaint analytics team.

Use only the complaint context provided below.
Do not make up information.
If the context is not enough, say: "The retrieved complaints do not provide enough information."

Context:
{context}

Question:
{question}

Answer format:
1. Main complaint theme:
2. Key issues identified:
3. Business risk:
4. Recommended action:
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json()["response"]

In [41]:
question = "What is the main problem in these credit report complaints?"

context = create_rag_context(question, top_k=5)

answer = ask_llm_with_prompt(context, question)

print(answer)

Based on the provided context, here are the answers to your question:

1. Main complaint theme: 
Inaccurate information reported on credit files, leading to potential data breaches and harm to consumers' credit profiles.

2. Key issues identified: 
- Inconsistent or inaccurate personal identifying information
- Unverifiable or missing information in credit reports
- Incorrect account balances or details
- Potential identity theft or unauthorized account activity
- Failure to properly investigate disputed accounts

3. Business risk:
The main business risk is the potential for reputational damage due to inaccurate credit reporting, which can lead to loss of customer trust and loyalty. The risk also extends to regulatory non-compliance with fair credit reporting act (FCRA), as evident in Complaint ID: 19922675.

4. Recommended action:
- Conduct thorough investigations into disputed accounts and credit reports
- Verify information through multiple sources to ensure accuracy
- Implement pro

In [45]:
def evaluate_rag(question):
    context = create_rag_context(question, top_k=5)
    answer = ask_llm_with_prompt(context, question)

    print("=" * 80)
    print("QUESTION")
    print(question)

    print("\n" + "=" * 80)
    print("RETRIEVED CONTEXT")
    print(context)

    print("\n" + "=" * 80)
    print("LLM ANSWER")
    print(answer)

In [47]:
evaluate_rag("What is the main issue in these complaints?")

QUESTION
What is the main issue in these complaints?

RETRIEVED CONTEXT

Complaint ID: 21471183
Complaint Text: detail complaint attached document
---

Complaint ID: 19523772
Complaint Text: complaint capital one high yield saving account major bank filing complaint unable open bank account institution despite valid explanation provided xx xx attempted open checking account application denied restricted without clear reasoning either told verbally qualify informed issue provided written documentation explaining specific reason denial right fair credit reporting act know consumer reporting agency used making decision report used request copy report specific reason denial decision based incorrect outdated information request investigated corrected immediately seeking clear written explanation denial copy consumer report used decision reconsideration application information inaccurate would like matter resolved soon possible sincerely
---

Complaint ID: 19291814
Complaint Text: actually c

In [49]:
evaluate_rag("What business risks are mentioned?")

QUESTION
What business risks are mentioned?

RETRIEVED CONTEXT

Complaint ID: 21702042
Complaint Text: xx xx year purchased industrial tool totaling upon delivery equipment found defective significant safety hazard specifically involving structural fracture motor housing initiated return request merchant day transaction well within stated day return policy ended xx xx year despite merchant refused return capital one since issued unauthorized rebill full amount capital one dispute department repeatedly ignored primary evidence exhibit prof timely outreach within policy window acted good faith resolve bank merchant currently forcing pay non functional dangerous equipment violation consumer protection standard merchant contract
---

Complaint ID: 21608364
Complaint Text: xx xx year also known florida provided contract promised would supply inventory store day wired chase bank may conducted adequate background check recipient bank account protected client participated money laundering frau